## Menghubungkan Google Drive dan Memuat Berkas Hasil Tahap Sebelumnya

In [1]:


import os
import json
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Mount Google Drive
drive.mount('/content/drive')

# Path Direktori
BASE_DRIVE_DIR = '/content/drive/MyDrive/CBR/'
EVAL_DIR = os.path.join(BASE_DRIVE_DIR, 'data/eval/')
RESULTS_DIR = os.path.join(BASE_DRIVE_DIR, 'data/results/')

# Load File Skenario Evaluasi (Ground Truth) dan Hasil Prediksi (Retrieval)
queries_json_path = os.path.join(EVAL_DIR, 'queries.json')
csv_predictions_path = os.path.join(RESULTS_DIR, 'predictions.csv')

if not os.path.exists(queries_json_path) or not os.path.exists(csv_predictions_path):
    print("[ERROR] File evaluasi atau prediksi tidak ditemukan! Pastikan Tahap 3 & 4 berhasil.")
else:
    with open(queries_json_path, 'r', encoding='utf-8') as f:
        ground_truth_data = json.load(f)

    df_predictions = pd.read_csv(csv_predictions_path)
    print(f"Data evaluasi berhasil dimuat. Siap menghitung performa model CBR!")

Mounted at /content/drive
Data evaluasi berhasil dimuat. Siap menghitung performa model CBR!


## Perhitungan Metrik Evaluasi Akhir

In [2]:
y_true = []
y_pred = []

# Loop mencocokkan setiap query uji
for query in ground_truth_data:
    q_id = query["query_id"]
    g_truth = query["ground_truth_case_id"][0] # Target ID kasus asli yang benar

    # Ambil baris hasil prediksi yang sesuai dengan query_id
    pred_row = df_predictions[df_predictions['query_id'] == q_id]

    if not pred_row.empty:
        # Ekstrak list top-5 case IDs dari string format csv
        top_5_str = pred_row.iloc[0]['top_5_case_ids']
        top_5_list = eval(top_5_str)

        # Jika masuk (Hit), prediksi sukses (1), jika tidak masuk dianggap (0)
        if g_truth in top_5_list:
            y_pred.append(1)
        else:
            y_pred.append(0)

        y_true.append(1)

# Menghitung metrik menggunakan scikit-learn
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# Menyusun hasil ke dalam DataFrame
metrics_summary = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Score_TF_IDF": [accuracy, precision, recall, f1]
}
df_metrics = pd.DataFrame(metrics_summary)

# Menyimpan hasil evaluasi ke dalam direktori sesuai regulasi tugas
retrieval_metrics_csv = os.path.join(EVAL_DIR, 'retrieval_metrics.csv')
df_metrics.to_csv(retrieval_metrics_csv, index=False, encoding='utf-8')

print("="*60)
print("LAPORAN PERFORMA RETRIEVAL MODEL (TF-IDF):")
print("="*60)
display(df_metrics)
print(f"\nFile laporan metrik sukses disimpan di: {retrieval_metrics_csv}")

LAPORAN PERFORMA RETRIEVAL MODEL (TF-IDF):


,Metric,Score_TF_IDF
0,Accuracy,0.0
1,Precision,0.0
2,Recall,0.0
3,F1-Score,0.0



File laporan metrik sukses disimpan di: /content/drive/MyDrive/CBR/data/eval/retrieval_metrics.csv


## Analisis Diskusi Kritis (Error & Rejection Analysis)

In [3]:
print("MELAKUKAN ERROR ANALYSIS (REJECTION DETECTION)...")
print("-" * 60)

failures = 0
for query in ground_truth_data:
    q_id = query["query_id"]
    g_truth = query["ground_truth_case_id"][0]

    pred_row = df_predictions[df_predictions['query_id'] == q_id]
    if not pred_row.empty:
        top_5_list = eval(pred_row.iloc[0]['top_5_case_ids'])

        if g_truth not in top_5_list:
            failures += 1
            print(f"KEGAGALAN TERDETEKSI PADA QUERY ID: {q_id}")
            print(f"   - Target Kasus Seharusnya: Case ID {g_truth}")
            print(f"   - Hasil Top-5 Model      : {top_5_list}")
            print(f"   - Potongan Teks Query    : {query['query_text'][:150]}...")
            print("-" * 50)

if failures == 0:
    print("Tidak ada kegagalan penempatan dokumen (Rejection = 0%).")
    print("Rekomendasi Perbaikan: Model TF-IDF saat ini sudah sangat baik untuk mengenali dokumen yang identik.")
else:
    print(f"Total terdapat {failures} kasus gagal temu.")
    print("Rekomendasi Perbaikan: Gunakan pendekatan Text Embedding (BERT/IndoBERT) untuk menangkap kesamaan semantik makna hukum.")

MELAKUKAN ERROR ANALYSIS (REJECTION DETECTION)...
------------------------------------------------------------
KEGAGALAN TERDETEKSI PADA QUERY ID: 31
   - Target Kasus Seharusnya: Case ID 31
   - Hasil Top-5 Model      : [10, 25, 32, 24, 30]
   - Potongan Teks Query    : : , tanggal 25 september 2024 yangamarnya berbunyi sebagai berikut:hal 2 dari 8 putusan no 825/pdt/2024/pt sby putusan.mahkamahagung.go.id mahkamah ag...
--------------------------------------------------
KEGAGALAN TERDETEKSI PADA QUERY ID: 19
   - Target Kasus Seharusnya: Case ID 19
   - Hasil Top-5 Model      : [8, 29, 33, 11, 6]
   - Potongan Teks Query    : tertanggal 24mei 2019 dan terdaftar di kepaniteraan pengadilan negeri surabaya tanggal31 mei 2019 dibawah register perkara no.552/pdt.g/2019/pn.sby., ...
--------------------------------------------------
KEGAGALAN TERDETEKSI PADA QUERY ID: 27
   - Target Kasus Seharusnya: Case ID 27
   - Hasil Top-5 Model      : [29, 33, 1, 11, 17]
   - Potongan Teks Query    :